In [ ]:
# Audio gahtering data ---> play all spikes then press yes for ball sound , and no if its any other sound
# Audio training SVM

!pip install -q librosa ipython
from google.colab import files
import librosa
import numpy as np
from IPython.display import Audio, display

print("Upload Sound.mp4")
uploaded = files.upload()
VIDEO_FILE = list(uploaded.keys())[0]

!ffmpeg -i "{VIDEO_FILE}" -q:a 0 -map a sound_audio.wav -y -loglevel quiet
y, sr = librosa.load("sound_audio.wav", sr=None)
print(f"Audio loaded: {sr}Hz, {len(y)/sr:.1f} seconds")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 16.6 MB/s eta 0:00:00
Upload Sound.mp4


In [ ]:
onset_env = librosa.onset.onset_strength(y=y, sr=sr)
onsets = librosa.onset.onset_detect(
    onset_envelope=onset_env,
    sr=sr, units='time',
    delta=0.5,
    wait=20
)
print(f"Found {len(onsets)} spikes")
print("Timestamps:", np.round(onsets, 2))

In [ ]:
!pip install -q ipywidgets
import ipywidgets as widgets
from IPython.display import Audio, display, clear_output
import numpy as np

CLIP_DURATION = 0.5
verified_spikes = []
current_idx = [0]

def show_spike(idx):
    clear_output(wait=True)
    if idx >= len(onsets):
        print(f"\n DONE! Verified {len(verified_spikes)} ball hits")
        print("Timestamps:", np.round(verified_spikes, 2))
        np.save("verified_spikes.npy", np.array(verified_spikes))
        print(" Saved → verified_spikes.npy")
        return

    t = onsets[idx]
    start  = max(0, t - 0.1)
    end    = min(len(y)/sr, t + CLIP_DURATION)
    clip   = y[int(start*sr):int(end*sr)]

    print(f"Spike {idx+1}/{len(onsets)} — Time: {t:.2f}s")
    display(Audio(clip, rate=sr, autoplay=True))  # auto plays!

    yes_btn = widgets.Button(description=" YES - Ball hit",
                              button_style='success', width='200px')
    no_btn  = widgets.Button(description=" NO - Not ball",
                              button_style='danger',  width='200px')

    def on_yes(b):
        verified_spikes.append(t)
        current_idx[0] += 1
        show_spike(current_idx[0])

    def on_no(b):
        current_idx[0] += 1
        show_spike(current_idx[0])

    yes_btn.on_click(on_yes)
    no_btn.on_click(on_no)
    display(widgets.HBox([yes_btn, no_btn]))

show_spike(0)

In [ ]:


from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
import numpy as np
import librosa
import pickle

CLIP_DURATION = 0.5
N_MFCC = 40

def extract_features(clip, sr, n_mfcc=N_MFCC):
    if len(clip) < sr * 0.1:
        return None
    mfcc = librosa.feature.mfcc(y=clip, sr=sr, n_mfcc=n_mfcc)
    chroma = librosa.feature.chroma_stft(y=clip, sr=sr)
    zcr = librosa.feature.zero_crossing_rate(clip)
    rmse = librosa.feature.rms(y=clip)
    feature_vec = np.concatenate([
        mfcc.mean(axis=1),
        mfcc.std(axis=1),
        chroma.mean(axis=1),
        zcr.mean(axis=1),
        rmse.mean(axis=1)
    ])
    return feature_vec


print("Extracting positive samples (ball hits)...")
X_pos = []
for t in verified_spikes:
    start = max(0, t - 0.1)
    end   = min(len(y)/sr, t + CLIP_DURATION)
    clip  = y[int(start*sr):int(end*sr)]
    feat  = extract_features(clip, sr)
    if feat is not None:
        X_pos.append(feat)

print(f" Positive samples: {len(X_pos)}")


print("Extracting negative samples (non-ball moments)...")
np.random.seed(42)
X_neg = []
duration = len(y) / sr


all_times = np.arange(1.0, duration - 1.0, 0.8)
non_spike_times = []
for t in all_times:
    if all(abs(t - s) > 1.0 for s in verified_spikes):
        non_spike_times.append(t)


selected_neg = np.random.choice(non_spike_times,
                                 size=min(len(X_pos), len(non_spike_times)),
                                 replace=False)

for t in selected_neg:
    start = max(0, t - 0.1)
    end   = min(len(y)/sr, t + CLIP_DURATION)
    clip  = y[int(start*sr):int(end*sr)]
    feat  = extract_features(clip, sr)
    if feat is not None:
        X_neg.append(feat)

print(f" Negative samples: {len(X_neg)}")

#Combine and train
X = np.array(X_pos + X_neg)
y_labels = np.array([1]*len(X_pos) + [0]*len(X_neg))

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


svm = SVC(kernel='rbf', C=10, gamma='scale', probability=True)
cv_scores = cross_val_score(svm, X_scaled, y_labels, cv=5)
print(f"\n Cross-validation accuracy: {cv_scores.mean()*100:.1f}% ± {cv_scores.std()*100:.1f}%")
print(f"   Scores per fold: {np.round(cv_scores*100, 1)}")


svm.fit(X_scaled, y_labels)
train_acc = svm.score(X_scaled, y_labels)
print(f"   Training accuracy: {train_acc*100:.1f}%")

In [ ]:


from google.colab import files

model_data = {
    "svm": svm,
    "scaler": scaler,
    "n_mfcc": N_MFCC,
    "clip_duration": CLIP_DURATION
}
with open("audio_svm.pkl", "wb") as f:
    pickle.dump(model_data, f)

print(" Binary SVM saved → audio_svm.pkl")
files.download("audio_svm.pkl")
files.download("verified_spikes.npy")

In [ ]:
from google.colab import files
import librosa

print("Upload all videos: T1.mp4, T2.mp4, T3.mp4, ST1.mp4, ST2.mp4, ST3.mp4, ST4.mp4, T6.mp4, T7.mp4, Sound.mp4")  # these videos are the videos we used in camera training
uploaded = files.upload()
video_files = list(uploaded.keys())
print(f" Uploaded {len(video_files)} videos: {video_files}")

In [ ]:
import librosa
import numpy as np
import os

all_new_spikes = []

for vid in video_files:
    print(f"\nProcessing {os.path.basename(vid)}...")
    audio_file = vid.replace(".mp4", "_audio.wav")
    os.system(f'ffmpeg -i "{vid}" -q:a 0 -map a "{audio_file}" -y -loglevel quiet')

    try:
        y_vid, sr_vid = librosa.load(audio_file, sr=None)
        print(f"  Duration: {len(y_vid)/sr_vid:.1f}s")

        onset_env_vid = librosa.onset.onset_strength(y=y_vid, sr=sr_vid)
        onsets_vid = librosa.onset.onset_detect(
            onset_envelope=onset_env_vid,
            sr=sr_vid, units='time',
            delta=0.5, wait=20
        )
        print(f"  Spikes found: {len(onsets_vid)}")
        for t in onsets_vid:
            all_new_spikes.append((os.path.basename(vid), t, y_vid, sr_vid))
    except Exception as e:
        print(f"   Error: {e}")

print(f"\n Total spikes to verify: {len(all_new_spikes)}")

In [ ]:
from google.colab import files
import zipfile, os, librosa


uploaded = files.upload()


zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall("tennis_videos")


video_files = []
for root, dirs, fnames in os.walk("tennis_videos"):
    for f in fnames:
        if f.endswith(".mp4"):
            video_files.append(os.path.join(root, f))

print(f" Found {len(video_files)} videos:")
for v in video_files:
    print(f"   {v}")

In [ ]:
import ipywidgets as widgets
from IPython.display import Audio, display, clear_output

new_verified = []
current_idx  = [0]

def show_new_spike(idx):
    clear_output(wait=True)
    if idx >= len(all_new_spikes):
        print(f"\n DONE! Verified {len(new_verified)} ball hits from 9 videos")
        print(f"Combined with 35 from Sound.mp4 = {len(new_verified) + 35} total positive samples")
        return

    vid_name, t, y_vid, sr_vid = all_new_spikes[idx]
    start = max(0, t - 0.1)
    end   = min(len(y_vid)/sr_vid, t + 0.5)
    clip  = y_vid[int(start*sr_vid):int(end*sr_vid)]

    # Progress bar
    progress = f"{idx+1}/{len(all_new_spikes)}"
    confirmed_so_far = len(new_verified)

    print(f"Spike {progress} | {vid_name} | Time: {t:.2f}s | Confirmed so far: {confirmed_so_far}")
    display(Audio(clip, rate=sr_vid, autoplay=True))

    yes_btn  = widgets.Button(description=" YES - Ball hit",   button_style='success', layout=widgets.Layout(width='180px'))
    no_btn   = widgets.Button(description=" NO - Not ball",    button_style='danger',  layout=widgets.Layout(width='180px'))
    skip_btn = widgets.Button(description="⏭ Skip this video",  button_style='warning', layout=widgets.Layout(width='180px'))

    def on_yes(b):
        new_verified.append((y_vid, sr_vid, t))
        current_idx[0] += 1
        show_new_spike(current_idx[0])

    def on_no(b):
        current_idx[0] += 1
        show_new_spike(current_idx[0])

    def on_skip(b):
        curr_vid = all_new_spikes[current_idx[0]][0]
        while current_idx[0] < len(all_new_spikes) and all_new_spikes[current_idx[0]][0] == curr_vid:
            current_idx[0] += 1
        show_new_spike(current_idx[0])

    yes_btn.on_click(on_yes)
    no_btn.on_click(on_no)
    skip_btn.on_click(on_skip)
    display(widgets.HBox([yes_btn, no_btn, skip_btn]))

show_new_spike(0)

In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report
import pickle
import os

N_MFCC = 40
CLIP_DURATION = 0.5

def extract_features(clip, sr):
    if len(clip) < sr * 0.1:
        return None
    try:
        mfcc   = librosa.feature.mfcc(y=clip, sr=sr, n_mfcc=N_MFCC)
        chroma = librosa.feature.chroma_stft(y=clip, sr=sr)
        zcr    = librosa.feature.zero_crossing_rate(clip)
        rmse   = librosa.feature.rms(y=clip)
        return np.concatenate([
            mfcc.mean(axis=1), mfcc.std(axis=1),
            chroma.mean(axis=1),
            zcr.mean(axis=1), rmse.mean(axis=1)
        ])
    except:
        return None


print("Reloading Sound.mp4 audio...")
os.system('ffmpeg -i "tennis_videos/VVV/Sound.mp4" -q:a 0 -map a sound_audio.wav -y -loglevel quiet')


sound_path = None
for root, dirs, fnames in os.walk("tennis_videos"):
    for f in fnames:
        if "sound" in f.lower() or "Sound" in f:
            sound_path = os.path.join(root, f)

if sound_path:
    os.system(f'ffmpeg -i "{sound_path}" -q:a 0 -map a sound_audio.wav -y -loglevel quiet')
    y_sound, sr_sound = librosa.load("sound_audio.wav", sr=None)
    print(f" Sound.mp4 loaded: {len(y_sound)/sr_sound:.1f}s")
else:
    print(" Sound.mp4 not in zip — using only 9 video spikes")
    y_sound = None

verified_spikes = np.load("verified_spikes.npy")


print("\nExtracting features from all positive samples...")
X_pos = []


if y_sound is not None:
    for t in verified_spikes:
        start = max(0, t - 0.1)
        end   = min(len(y_sound)/sr_sound, t + CLIP_DURATION)
        clip  = y_sound[int(start*sr_sound):int(end*sr_sound)]
        feat  = extract_features(clip, sr_sound)
        if feat is not None:
            X_pos.append(feat)
    print(f"  From Sound.mp4: {len(X_pos)} samples")


count_before = len(X_pos)
for y_vid, sr_vid, t in new_verified:
    start = max(0, t - 0.1)
    end   = min(len(y_vid)/sr_vid, t + CLIP_DURATION)
    clip  = y_vid[int(start*sr_vid):int(end*sr_vid)]
    feat  = extract_features(clip, sr_vid)
    if feat is not None:
        X_pos.append(feat)
print(f"  From 9 videos : {len(X_pos) - count_before} samples")
print(f"  Total positive: {len(X_pos)}")


print("\nExtracting negative samples...")
X_neg = []
np.random.seed(42)


audio_sources = []
if y_sound is not None:
    audio_sources.append((y_sound, sr_sound, verified_spikes.tolist()))


from collections import defaultdict
vid_spike_map = defaultdict(list)
for y_vid, sr_vid, t in new_verified:

    vid_spike_map[id(y_vid)].append((y_vid, sr_vid, t))

for vid_id, items in vid_spike_map.items():
    y_v, sr_v = items[0][0], items[0][1]
    spike_times = [item[2] for item in items]
    audio_sources.append((y_v, sr_v, spike_times))


neg_per_source = max(5, len(X_pos) // len(audio_sources))
for y_v, sr_v, spike_times in audio_sources:
    duration = len(y_v) / sr_v
    all_times = np.arange(1.0, duration - 1.0, 0.8)
    non_spike = [t for t in all_times if all(abs(t - s) > 1.0 for s in spike_times)]
    if len(non_spike) == 0:
        continue
    n_neg = min(neg_per_source, len(non_spike))
    selected = np.random.choice(non_spike, size=n_neg, replace=False)
    for t in selected:
        start = max(0, t - 0.1)
        end   = min(len(y_v)/sr_v, t + CLIP_DURATION)
        clip  = y_v[int(start*sr_v):int(end*sr_v)]
        feat  = extract_features(clip, sr_v)
        if feat is not None:
            X_neg.append(feat)

print(f"  Total negative: {len(X_neg)}")


X        = np.array(X_pos + X_neg)
y_labels = np.array([1]*len(X_pos) + [0]*len(X_neg))

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

svm = SVC(kernel='rbf', C=10, gamma='scale', probability=True)


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(svm, X_scaled, y_labels, cv=cv)
svm.fit(X_scaled, y_labels)

print(f"\n{'='*50}")
print(f"AUDIO SVM RESULTS")
print(f"  Positive samples : {len(X_pos)}")
print(f"  Negative samples : {len(X_neg)}")
print(f"  CV accuracy      : {cv_scores.mean()*100:.1f}% ± {cv_scores.std()*100:.1f}%")
print(f"  Training accuracy: {svm.score(X_scaled, y_labels)*100:.1f}%")
print(f"{'='*50}")

In [ ]:


import numpy as np


verified_timestamps = np.array([t for y_vid, sr_vid, t in new_verified])
np.save("verified_spikes.npy", verified_timestamps)
print(f"✅ verified_spikes.npy rebuilt: {len(verified_timestamps)} spikes")
print("Timestamps:", np.round(verified_timestamps[:10], 2), "...")

In [ ]:

verified_spikes = np.load("verified_spikes.npy")


y_sound = None

In [ ]:
model_data = {
    "svm":           svm,
    "scaler":        scaler,
    "n_mfcc":        N_MFCC,
    "clip_duration": CLIP_DURATION
}
with open("audio_svm.pkl", "wb") as f:
    pickle.dump(model_data, f)

print(" Audio SVM model saved!")
files.download("audio_svm.pkl")
files.download("verified_spikes.npy")

In [ ]:


import os
import librosa
import numpy as np
import pickle
import matplotlib.pyplot as plt
from google.colab import files

with open("audio_svm.pkl", "rb") as f:
    model_data = pickle.load(f)

svm_audio      = model_data["svm"]
scaler_audio   = model_data["scaler"]
N_MFCC         = model_data["n_mfcc"]
CLIP_DURATION  = model_data["clip_duration"]

print(" Audio SVM loaded")


print("\nUpload your test video (e.g. your FF7/TEST video)...")
uploaded = files.upload()
test_video = list(uploaded.keys())[0]
print(f" Video uploaded: {test_video}")


os.system(f'ffmpeg -i "{test_video}" -q:a 0 -map a test_audio.wav -y -loglevel quiet')
y_test, sr_test = librosa.load("test_audio.wav", sr=None)
duration_test   = len(y_test) / sr_test
VIDEO_FPS       = 61.9

print(f"Audio duration : {duration_test:.1f}s")
print(f"Video FPS      : {VIDEO_FPS}")


def extract_features(clip, sr):
    if len(clip) < sr * 0.1:
        return None
    try:
        mfcc   = librosa.feature.mfcc(y=clip, sr=sr, n_mfcc=N_MFCC)
        chroma = librosa.feature.chroma_stft(y=clip, sr=sr)
        zcr    = librosa.feature.zero_crossing_rate(clip)
        rmse   = librosa.feature.rms(y=clip)
        return np.concatenate([
            mfcc.mean(axis=1), mfcc.std(axis=1),
            chroma.mean(axis=1),
            zcr.mean(axis=1), rmse.mean(axis=1)
        ])
    except:
        return None


STEP           = 0.1
MIN_GAP        = 1.0
PROB_THRESHOLD = 0.75

detections     = []
last_detection = -MIN_GAP
probs_all      = []
times_all      = []

print("\nRunning audio SVM detector...")

t = CLIP_DURATION / 2
while t < duration_test - CLIP_DURATION / 2:
    start = max(0, t - 0.1)
    end   = min(duration_test, t + CLIP_DURATION)
    clip  = y_test[int(start * sr_test):int(end * sr_test)]

    feat = extract_features(clip, sr_test)
    if feat is not None:
        feat_scaled = scaler_audio.transform([feat])
        prob = svm_audio.predict_proba(feat_scaled)[0][1]
        probs_all.append(prob)
        times_all.append(t)

        if prob >= PROB_THRESHOLD and (t - last_detection) >= MIN_GAP:
            frame_num = int(t * VIDEO_FPS)
            detections.append({
                "time_sec"  : round(t, 2),
                "frame"     : frame_num,
                "confidence": round(prob, 3)
            })
            last_detection = t
            print(f"   Ball hit: t={t:.2f}s | frame={frame_num:5d} | conf={prob:.3f}")

    t += STEP


print(f"\n{'='*55}")
print(f"AUDIO STANDALONE RESULTS")
print(f"  Video          : {test_video}")
print(f"  Duration       : {duration_test:.1f}s")
print(f"  Ball hits found: {len(detections)}")
print(f"  Timestamps(s)  : {[d['time_sec'] for d in detections]}")
print(f"  Frame numbers  : {[d['frame'] for d in detections]}")
print(f"{'='*55}")


plt.figure(figsize=(14, 4))
plt.plot(times_all, probs_all, color='steelblue', alpha=0.7, label='Ball hit probability')
plt.axhline(y=PROB_THRESHOLD, color='red', linestyle='--', label=f'Threshold ({PROB_THRESHOLD})')
for d in detections:
    plt.axvline(x=d['time_sec'], color='green', alpha=0.8, linewidth=2)
plt.xlabel("Time (seconds)")
plt.ylabel("SVM Probability")
plt.title(f"Audio SVM — {len(detections)} ball hits detected (green lines)")
plt.legend()
plt.tight_layout()
plt.savefig("audio_standalone_result.png", dpi=100)
plt.show()
files.download("audio_standalone_result.png")


audio_detections = np.array([[d['time_sec'], d['frame'], d['confidence']]
                               for d in detections])
np.save("audio_detections.npy", audio_detections)
print(" Saved → audio_detections.npy")
files.download("audio_detections.npy")